# ScamShield BD — Phase 1 Baseline Solution
**Team Name:** Shield

## Declaration
- **Required Libraries & Versions:** `python==3.14`, `numpy==2.5.1`, `pandas==3.1.1`, `scipy==1.18.1`, `scikit-learn==1.9.1`, `python-dotenv==1.2.4`
- **External Datasets:** None (trained strictly on `sample.csv`).
- **Pretrained Models:** None.
- **APIs Used:** None (no API calls; `.env` is empty).
- **Methodology & Validation Strategy:**
  - *Preprocessing:* Unicode NFKC normalization; removal of `\u200b`, `\u200c`, `\ufeff` (preserving `\u200d` ZWJ for Bangla); Cyrillic/Greek-to-Latin homoglyph mapping; Latin-only lowercasing; defanged URL replacement (`[URL]`); digit-run replacement (`[NUM]`); `|||` conversation separator preservation.
  - *Features:* `TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True)` + `TfidfVectorizer(analyzer='word', ngram_range=(1, 2), sublinear_tf=True)` combined via `scipy.sparse.hstack`.
  - *Models:* Task A (`LogisticRegression(class_weight='balanced')`), Task B (`OneVsRestClassifier(LogisticRegression(class_weight='balanced'))`, threshold `0.5`), Task C (`LogisticRegression(class_weight='balanced')`).
  - *Post-Processing:* `label == 0 -> techniques = 'NONE', action = 0`; non-zero labels with empty technique sets fall back to the single highest-probability technique; sorted `|`-joined codes; confidence = max predicted Task A probability clipped to `[0.05, 0.98]` and rounded to 4 decimals.
  - *Validation:* 5-fold `StratifiedKFold` on `sample.csv` (note: CV is optimistic due to near-duplicate templates).
- **Random Seeds:** `SEED = 42` for `random`, `numpy`, and `sklearn`.
- **Data Augmentation Code:** None at this baseline stage.
- **How to Reproduce:** Place `sample.csv`, `test.csv`, and `sample_submission.csv` in the working directory (or `./data/`) and run all cells top to bottom.


In [1]:
import os
import re
import time
import random
import unicodedata
import numpy as np
import pandas as pd
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from dotenv import load_dotenv

load_dotenv()

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = 'data' if os.path.isdir('data') else '.'
TECH_CODES = ['T1', 'T2', 'T3', 'T4', 'T5', 'T6']
print('Environment ready. SEED =', SEED, '| DATA_DIR =', DATA_DIR)


Environment ready. SEED = 42 | DATA_DIR = .


## 1. Load Data

In [2]:
sample = pd.read_csv(os.path.join(DATA_DIR, 'sample.csv'))
test_meta = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'), usecols=['id'])
sample_sub = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'), usecols=['id'])
print('sample.csv shape:', sample.shape)
print('test.csv rows:', len(test_meta))
print('sample_submission.csv rows:', len(sample_sub))


sample.csv shape: (86, 5)
test.csv rows: 500000
sample_submission.csv rows: 500000


## 2. Preprocessing

In [3]:
HOMOGLYPH_MAP = {
    '\u0430': 'a', '\u0435': 'e', '\u0451': 'e', '\u0456': 'i', '\u0457': 'i',
    '\u0458': 'j', '\u043e': 'o', '\u0440': 'p', '\u0441': 'c', '\u0443': 'y',
    '\u0445': 'x', '\u0455': 's', '\u0501': 'd', '\u051b': 'q', '\u051d': 'w',
    '\u0410': 'A', '\u0412': 'B', '\u0415': 'E', '\u041a': 'K', '\u041c': 'M',
    '\u041d': 'H', '\u041e': 'O', '\u0420': 'P', '\u0421': 'C', '\u0422': 'T',
    '\u0425': 'X', '\u04ae': 'Y', '\u0406': 'I',
    '\u03b1': 'a', '\u03b5': 'e', '\u03b9': 'i', '\u03bf': 'o', '\u03c1': 'p',
    '\u03c2': 'c', '\u0391': 'A', '\u0392': 'B', '\u0395': 'E', '\u0397': 'H',
    '\u0399': 'I', '\u039a': 'K', '\u039c': 'M', '\u039d': 'N', '\u039f': 'O',
    '\u03a1': 'P', '\u03a4': 'T', '\u03a7': 'X',
}
HOMOGLYPH_TRANS = str.maketrans(HOMOGLYPH_MAP)
URL_PATTERN = re.compile(r'(?:hxxps?|https?|ftp)://\S+|(?:[a-zA-Z0-9-]+\[\.\])+[a-zA-Z]{2,}(?:/\S*)?', re.IGNORECASE)
DIGIT_RUN_PATTERN = re.compile(r'[0-9\u09e6-\u09ef]+(?:[.,][0-9\u09e6-\u09ef]+)*')
LATIN_UPPER_PATTERN = re.compile(r'[A-Z]+')

def preprocess_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    t = unicodedata.normalize('NFKC', text)
    t = t.replace('\u200b', '').replace('\u200c', '').replace('\ufeff', '')
    t = t.translate(HOMOGLYPH_TRANS)
    t = URL_PATTERN.sub(' [URL] ', t)
    t = LATIN_UPPER_PATTERN.sub(lambda m: m.group(0).lower(), t)
    t = t.replace('[url]', '[URL]')
    t = DIGIT_RUN_PATTERN.sub(' [NUM] ', t)
    t = t.replace('|||', ' ||| ')
    t = re.sub(r'\s+', ' ', t).strip()
    return t


## 3. Baseline Training & 5-Fold Stratified CV

In [4]:
def encode_techniques(series: pd.Series) -> np.ndarray:
    y = np.zeros((len(series), len(TECH_CODES)), dtype=int)
    for i, val in enumerate(series.astype(str)):
        parts = val.split('|')
        for j, code in enumerate(TECH_CODES):
            if code in parts:
                y[i, j] = 1
    return y

def postprocess_predictions(labels, tech_probs, actions, confs):
    out_techs = []
    out_actions = actions.copy()
    for i in range(len(labels)):
        if labels[i] == 0:
            out_techs.append('NONE')
            out_actions[i] = 0
        else:
            active = [TECH_CODES[j] for j in range(len(TECH_CODES)) if tech_probs[i, j] >= 0.5]
            if not active:
                active = [TECH_CODES[int(np.argmax(tech_probs[i]))]]
            out_techs.append('|'.join(sorted(active)))
    out_confs = np.round(np.clip(confs, 0.05, 0.98), 4)
    return labels, out_techs, out_actions, out_confs

def fit_vectorizers(texts):
    char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True, dtype=np.float32)
    word_vec = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), sublinear_tf=True, dtype=np.float32)
    return char_vec, word_vec, hstack([char_vec.fit_transform(texts), word_vec.fit_transform(texts)]).tocsr()

def transform_vectorizers(char_vec, word_vec, texts):
    return hstack([char_vec.transform(texts), word_vec.transform(texts)]).tocsr()

texts = sample['text'].fillna('').astype(str).apply(preprocess_text).tolist()
y_label = sample['label'].astype(int).values
y_tech = encode_techniques(sample['techniques'])
y_action = sample['action'].astype(int).values

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_label = np.zeros(len(sample), dtype=int)
oof_tech_probs = np.zeros_like(y_tech, dtype=float)
oof_action = np.zeros(len(sample), dtype=int)
oof_conf = np.zeros(len(sample), dtype=float)

for tr_idx, va_idx in skf.split(texts, y_label):
    cv_c, cv_w, X_tr = fit_vectorizers([texts[i] for i in tr_idx])
    X_va = transform_vectorizers(cv_c, cv_w, [texts[i] for i in va_idx])
    ca = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED).fit(X_tr, y_label[tr_idx])
    pa = ca.predict_proba(X_va)
    oof_label[va_idx] = np.argmax(pa, axis=1)
    oof_conf[va_idx] = np.max(pa, axis=1)
    cb = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)).fit(X_tr, y_tech[tr_idx])
    oof_tech_probs[va_idx] = cb.predict_proba(X_va)
    cc = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED).fit(X_tr, y_action[tr_idx])
    oof_action[va_idx] = cc.predict(X_va)

oof_label, oof_tech_strs, oof_action, oof_conf = postprocess_predictions(oof_label, oof_tech_probs, oof_action, oof_conf)
oof_tech_bin = encode_techniques(pd.Series(oof_tech_strs))
print(f'CV Task A Macro F1: {f1_score(y_label, oof_label, average="macro"):.4f}')
print(f'CV Task B Macro F1: {np.mean([f1_score(y_tech[:, j], oof_tech_bin[:, j], zero_division=0) for j in range(6)]):.4f}')
print(f'CV Task C Macro F1: {f1_score(y_action, oof_action, average="macro"):.4f}')


CV Task A Macro F1: 0.5828
CV Task B Macro F1: 0.4463
CV Task C Macro F1: 0.4940


## 4. Train on Full `sample.csv`, Predict `test.csv`, & Post-Process

In [5]:
t0 = time.time()
char_vec, word_vec, X_full = fit_vectorizers(texts)
full_clf_a = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED).fit(X_full, y_label)
full_clf_b = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED)).fit(X_full, y_tech)
full_clf_c = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED).fit(X_full, y_action)

out_path = 'submission.csv'
first_chunk = True
for chunk in pd.read_csv(os.path.join(DATA_DIR, 'test.csv'), chunksize=50000):
    clean_batch = chunk['text'].fillna('').astype(str).apply(preprocess_text).tolist()
    X_test = transform_vectorizers(char_vec, word_vec, clean_batch)
    probs_a = full_clf_a.predict_proba(X_test)
    preds_a = np.argmax(probs_a, axis=1)
    confs = np.max(probs_a, axis=1)
    tech_probs = full_clf_b.predict_proba(X_test)
    preds_c = full_clf_c.predict(X_test)
    preds_a, tech_strs, preds_c, confs = postprocess_predictions(preds_a, tech_probs, preds_c, confs)
    pd.DataFrame({
        'id': chunk['id'].values.astype(int),
        'label': preds_a.astype(int),
        'techniques': tech_strs,
        'action': preds_c.astype(int),
        'confidence': confs,
    }).to_csv(out_path, mode='w' if first_chunk else 'a', header=first_chunk, index=False)
    first_chunk = False
print(f'Wrote {out_path} in {time.time()-t0:.2f}s')


Wrote submission.csv in 92.68s


## 5. Validate `submission.csv`

In [6]:
REQUIRED_COLS = ['id', 'label', 'techniques', 'action', 'confidence']
VALID_CODES = {'T1', 'T2', 'T3', 'T4', 'T5', 'T6'}
def check_tech(val):
    if not isinstance(val, str) or val != val.strip() or not val or ' ' in val:
        return False
    if val == 'NONE':
        return True
    parts = val.split('|')
    return 'NONE' not in parts and all(p in VALID_CODES for p in parts) and len(parts) == len(set(parts)) and parts == sorted(parts)

sub = pd.read_csv('submission.csv', keep_default_na=False)
assert list(sub.columns) == REQUIRED_COLS
assert len(sub) == len(test_meta) == len(sample_sub)
assert (sub['id'].astype(int).values == test_meta['id'].values).all()
assert not sub['id'].duplicated().any()
for col in REQUIRED_COLS:
    assert (sub[col].astype(str).str.strip() != '').all()
assert sub['label'].astype(int).isin([0, 1, 2, 3, 4, 5]).all()
assert sub['action'].astype(int).isin([0, 1, 2, 3, 4, 5]).all()
assert sub['techniques'].apply(check_tech).all()
confs = pd.to_numeric(sub['confidence'], errors='raise')
assert not confs.isna().any() and ((confs >= 0.0) & (confs <= 1.0)).all()
print('PASS: submission.csv validated successfully! Rows:', len(sub))


PASS: submission.csv validated successfully! Rows: 500000
